In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("quikr_car.csv")

In [3]:
data.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [4]:
data.shape

(892, 6)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        892 non-null    object
 1   company     892 non-null    object
 2   year        892 non-null    object
 3   Price       892 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: object(6)
memory usage: 41.9+ KB


In [6]:
backup=data.copy()

## Quality Issues

- names are pretty inconsistent
- names have company names attached to it
- some names are spam like 'Maruti Ertiga showroom condition with' and 'Well mentained Tata Sumo'
- company: many of the names are not of any company like 'Used', 'URJENT', and so on.
- year has many non-year values
- year is in object. Change to integer
- Price has Ask for Price
- Price has commas in its prices and is in object
- kms_driven has object values with kms at last.
- It has nan values and two rows have 'Petrol' in them
- fuel_type has nan values

## Starting Cleaning Data 

- #### year has many non-year values

In [7]:
data = data[data['year'].str.isnumeric()]

- #### year is in object. Change to integer

In [8]:
data['year']=data['year'].astype(int)

- #### Price cotains  "has Ask for Price"

In [9]:
data=data[data['Price']!='Ask For Price']

- #### Price has commas in its values and its dataType is object

In [10]:
data['Price']=data['Price'].str.replace(',','').astype(int)

- ####  kms_driven has dataType object and have "kms" at last.

In [11]:
data['kms_driven']=data['kms_driven'].str.split().str.get(0).str.replace(',','')

- #### kms_driven column  has nan values and two rows have value 'Petrol' in them

In [12]:
data = data[data['kms_driven'].str.isnumeric()]

- #### kms_driven has dataType of Object

In [13]:
data['kms_driven']=data['kms_driven'].astype(int)

- #### fuel_type has nan values

In [14]:
data=data[~data['fuel_type'].isna()]

In [15]:
data.shape

(816, 6)

In [16]:
data=data.reset_index(drop=True)

## Cleaned Data with corrected Index

In [17]:
data

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,425000,40,Diesel
2,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,325000,28000,Petrol
3,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,575000,36000,Diesel
4,Ford Figo,Ford,2012,175000,41000,Diesel
...,...,...,...,...,...,...
811,Maruti Suzuki Ritz VXI ABS,Maruti,2011,270000,50000,Petrol
812,Tata Indica V2 DLE BS III,Tata,2009,110000,30000,Diesel
813,Toyota Corolla Altis,Toyota,2009,300000,132000,Petrol
814,Tata Zest XM Diesel,Tata,2018,260000,27000,Diesel


In [18]:
data.to_csv('Cleaned_Car_data.csv')

In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 816 entries, 0 to 815
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        816 non-null    object
 1   company     816 non-null    object
 2   year        816 non-null    int64 
 3   Price       816 non-null    int64 
 4   kms_driven  816 non-null    int64 
 5   fuel_type   816 non-null    object
dtypes: int64(3), object(3)
memory usage: 38.4+ KB


In [20]:
data.describe(include='all')

,name,company,year,Price,kms_driven,fuel_type
count,816,816,816.000000,8.160000e+02,816.000000,816
unique,463,25,NaN,NaN,NaN,3
top,Honda City,Maruti,NaN,NaN,NaN,Petrol
freq,13,221,NaN,NaN,NaN,428
mean,NaN,NaN,2012.444853,4.117176e+05,46275.531863,NaN
std,NaN,NaN,4.002992,4.751844e+05,34297.428044,NaN
min,NaN,NaN,1995.000000,3.000000e+04,0.000000,NaN
25%,NaN,NaN,2010.000000,1.750000e+05,27000.000000,NaN
50%,NaN,NaN,2013.000000,2.999990e+05,41000.000000,NaN
75%,NaN,NaN,2015.000000,4.912500e+05,56818.500000,NaN


In [21]:
data=data[data['Price']<6000000]

### Extracting Training Data

In [22]:
X=data[['name','company','year','kms_driven','fuel_type']]
y=data['Price']

### Applying Train Test Split

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=655
)

In [24]:
from sklearn.linear_model import LinearRegression

In [25]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

In [26]:
ohe=OneHotEncoder()
ohe.fit(X[['name','company','fuel_type']])
OneHotEncoder()

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [27]:
column_trans=make_column_transformer((OneHotEncoder(categories=ohe.categories_,handle_unknown='ignore'),['name','company','fuel_type']),
                                    remainder='passthrough')

### Train Model

In [28]:
model=LinearRegression()

### Create Pipeline

In [29]:
pipe=make_pipeline(column_trans,model)

In [30]:
pipe.fit(X_train,y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [31]:
y_pred=pipe.predict(X_test)

In [32]:
r2_score(y_test,y_pred)

0.6278534319121843

In [33]:
import numpy as np

In [34]:
data["company"].unique()

array(['Hyundai', 'Mahindra', 'Ford', 'Maruti', 'Skoda', 'Audi', 'Toyota',
       'Renault', 'Honda', 'Datsun', 'Mitsubishi', 'Tata', 'Volkswagen',
       'Chevrolet', 'Mini', 'BMW', 'Nissan', 'Hindustan', 'Fiat', 'Force',
       'Mercedes', 'Land', 'Jaguar', 'Jeep', 'Volvo'], dtype=object)

In [35]:
#data["name"].unique()

In [36]:
pipe.predict(pd.DataFrame(columns=X_test.columns,data=np.array(['Toyota Fortuner','Toyota',2025,1000,'Petrol']).reshape(1,5)))

array([904028.84687793])

In [37]:
import json
import pickle

# 1. Export the dropdown JSON for React mapping
df = pd.read_csv('Cleaned_Car_data.csv')
company_car_map = {}
for company in df['company'].unique():
    company_car_map[company] = sorted(df[df['company'] == company]['name'].unique().tolist())

dropdown_data = {
    'companies': sorted(df['company'].unique().tolist()),
    'company_car_map': company_car_map,
    'fuel_types': sorted(df['fuel_type'].unique().tolist()),
    'years': sorted(df['year'].unique().tolist(), reverse=True)
}

with open('dropdown_options.json', 'w') as f:
    json.dump(dropdown_data, f)

# 2. Export the trained Pipeline model
with open('LinearRegressionModel.pkl', 'wb') as f:
    pickle.dump(pipe, f)

print("🎉 Success! 'dropdown_options.json' and 'LinearRegressionModel.pkl' generated.")

🎉 Success! 'dropdown_options.json' and 'LinearRegressionModel.pkl' generated.
